In [ ]:
#!/usr/bin/env python3
import os
import tensorflow as tf
import numpy as np
import random
import datetime
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time
import math
import io
from collections import deque # use queue for replay buffer

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
 try:
  for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
  logical_gpus = tf.config.experimental.list_logical_devices('GPU')
  print(len(gpus), "Physical GPUs,", len(logical_gpus), "GPUs configured with mem growth")
 except RuntimeError as e:
   print(f"mem growth error: {e}")
else:
   print("No GPU detected by TF")

# converts a flat index to (x, y, z) coordinates.
@tf.function
def flat_to_xyz(flat_idx, G):
    g_tf = tf.cast(G, tf.int32)
    z = flat_idx % g_tf
    y = (flat_idx // g_tf) % g_tf
    x = flat_idx // (g_tf * g_tf)
    # ensure shapes match: if flat_idx is [N], result is [N, 3]
    if tf.rank(flat_idx) == 0: # Single index
        return tf.stack([x, y, z], axis=-1)
    else:
        return tf.stack([x, y, z], axis=-1)

# converts (x, y, z) coordinates to a flat index.
@tf.function
def xyz_to_flat(x, y, z, G):
    g_tf = tf.cast(G, tf.int32)
    # all inputs gotta have same shape
    return x * g_tf * g_tf + y * g_tf + z

# (x, y, z) coordinates must be within [0, G-1]
@tf.function
def clamp_pos_tf(pos_xyz, G):
    g_minus_1 = tf.cast(G - 1, tf.int32)
    return tf.clip_by_value(pos_xyz, 0, g_minus_1)

# Manhattan distance between two sets of (x, y, z) coordinates
@tf.function
def calculate_manhattan_distance(pos1_xyz, pos2_xyz):
    return tf.reduce_sum(tf.abs(pos1_xyz - pos2_xyz), axis=-1)

# convert a batch of discrete goal indices to relative vectors.
@tf.function
def goal_index_to_delta_xyz(goal_index_batch, k):
    # goal_index_batch: [N]
    # returns: delta_xyz_batch [N, 3]
    base = tf.cast(2 * k + 1, tf.int32)
    shift = tf.constant(k, dtype=tf.int32)

    dz_batch = goal_index_batch % base
    dy_batch = tf.cast((goal_index_batch // base) % base, tf.int32)
    dx_batch = tf.cast(goal_index_batch // (base * base), tf.int32)

    # shift from [0, 2k] range to [-k, k] range
    return tf.stack([dx_batch - shift, dy_batch - shift, dz_batch - shift], axis=-1)

# calculates the target flat position for the subgoal based on start position and goal.
@tf.function
def calculate_subgoal_target_pos_flat(start_pos_flat_batch, goal_index_batch, G, k):
    # start_pos_flat_batch: [N]
    # goal_index_batch: [N]
    # returns: target_pos_flat_batch [N]

    start_pos_xyz_batch = flat_to_xyz(start_pos_flat_batch, G)
    delta_xyz_batch = goal_index_to_delta_xyz(goal_index_batch, k)

    target_xyz_raw_batch = start_pos_xyz_batch + delta_xyz_batch
    target_xyz_clamped_batch = clamp_pos_tf(target_xyz_raw_batch, G)

    target_pos_flat_batch = xyz_to_flat(target_xyz_clamped_batch[:, 0],
                                        target_xyz_clamped_batch[:, 1],
                                        target_xyz_clamped_batch[:, 2], G)
    return target_pos_flat_batch

# intrinsic reward for a batch based on progress towards subgoal target.
@tf.function
def calculate_intrinsic_reward_batch(pos_t_flat_batch, pos_tplus1_flat_batch, subgoal_start_pos_flat_batch, current_goal_index_batch, G, k):
    # pos_t_flat_batch: [N] - flat position before the step
    # pos_tplus1_flat_batch: [N] - flat position after the step
    # subgoal_start_pos_flat_batch: [N] - flat position where the current goal was set
    # current_goal_index_batch: [N] - the goal index set by the manager

    target_pos_flat_batch = calculate_subgoal_target_pos_flat(
        subgoal_start_pos_flat_batch, current_goal_index_batch, G, k
    )

    # convert target_pos_flat_batch to xyz for distance calculation
    target_pos_xyz_batch = flat_to_xyz(target_pos_flat_batch, G)

    pos_t_xyz_batch = flat_to_xyz(pos_t_flat_batch, G)
    pos_tplus1_xyz_batch = flat_to_xyz(pos_tplus1_flat_batch, G)


    dist_t = calculate_manhattan_distance(pos_t_xyz_batch, target_pos_xyz_batch)
    dist_tplus1 = calculate_manhattan_distance(pos_tplus1_xyz_batch, target_pos_xyz_batch)

    # intrinsic reward: decrease in distance
    intrinsic_reward_batch = tf.cast(dist_t, tf.float32) - tf.cast(dist_tplus1, tf.float32)

    return intrinsic_reward_batch


# Custom environment class
class BatchedSculpt3DEnvTF:
    def __init__(self, grid_size=16, max_steps=200, n_envs=16):
        G, N = grid_size, n_envs
        if N <= 0: raise ValueError("n_envs must be positive.")
        self.G, self.N, self.max_steps = G, N, max_steps
        self.flat_dim = G*G*G
        self.grid_obs_shape = (G, G, G, 2) # channels: Stock, ShapeMask
        self.coord_obs_shape = (3,)        # channels: X, Y, Z (normalized)

        coords_range = tf.range(G, dtype=tf.float32)
        coords = tf.stack(tf.meshgrid(coords_range, coords_range, coords_range, indexing='ij'), axis=-1)
        center = tf.constant([G/2 - 0.5, G/2 - 0.5, G/2 - 0.5], tf.float32)
        dist2 = tf.reduce_sum(tf.square(coords - center), axis=-1)
        radius_sq = tf.square(tf.cast(G // 2 - 1, tf.float32))
        mask3d = dist2 <= radius_sq
        mask_flat = tf.reshape(mask3d, [-1])

        self.shape_mask = tf.Variable(tf.tile(mask_flat[None, :], [N, 1]), trainable=False, dtype=tf.bool, name="shape_mask")
        self.stock = tf.Variable(tf.ones([N, self.flat_dim], dtype=tf.bool), trainable=False, name="stock")
        self.pos = tf.Variable(tf.zeros([N], dtype=tf.int32), trainable=False, name="pos")
        self.steps = tf.Variable(tf.zeros([N], dtype=tf.int32), trainable=False, name="steps")
        self.done = tf.Variable(tf.zeros([N], dtype=tf.bool), trainable=False, name="done")

        G_py = grid_size
        def to_flat_py(dx, dy, dz): return dx*G_py*G_py + dy*G_py + dz
        moves = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]
        shifts_py = [to_flat_py(*m) for m in moves]
        self.shifts = tf.constant(shifts_py, dtype=tf.int32, name="shifts")


    @tf.function
    def reset(self):
        # reset stock to full
        self.stock.assign(tf.ones_like(self.stock))
        self.steps.assign(tf.zeros_like(self.steps))
        self.done.assign(tf.zeros_like(self.done))

        # find save start position
        safe_indices = tf.where(tf.logical_not(self.shape_mask[0]))[:, 0]
        num_safe = tf.shape(safe_indices)[0]
        tf.debugging.assert_greater_equal(num_safe, self.N, message="Not enough safe starting positions available.")

        shuffled_safe_indices = tf.random.shuffle(safe_indices)[:self.N]
        self.pos.assign(tf.cast(shuffled_safe_indices, tf.int32))

        return self._get_obs()

    @tf.function
    def step(self, actions):
        # calc potential new positions based on actions
        action_shifts = tf.gather(self.shifts, actions)
        new_pos = self.pos + action_shifts

        # ckeck boundaries and collisions
        in_bounds = tf.logical_and(new_pos >= 0, new_pos < self.flat_dim)
        safe_new_pos = tf.clip_by_value(new_pos, 0, self.flat_dim - 1)

        # create shape mask and stock at the potential new positions
        shape_mask_at_new = tf.gather(self.shape_mask, safe_new_pos, axis=1, batch_dims=1)
        stock_at_new = tf.gather(self.stock, safe_new_pos, axis=1, batch_dims=1)

        # determine invalid moves (hit shape or out of bounds)
        hit_shape_or_oob = tf.logical_or(tf.logical_not(in_bounds), shape_mask_at_new)

        # determine if stock can be removed (valid move AND stock exists at new pos)
        can_remove = tf.logical_and(tf.logical_not(hit_shape_or_oob), stock_at_new)

        # grant rewards
        reward = tf.where(hit_shape_or_oob, -5.0, 0.0) # invalid move penalty
        reward = tf.where(can_remove, reward + 1.0, reward) # reward for removing stock
        reward = reward - 0.1                           # step penalty (encouirage optimal toolpath)


        # update stock (remove material where applicable)
        remove_indices = tf.where(can_remove)
        num_removals = tf.shape(remove_indices)[0]

        # conditional update to avoid empty tensor issues if num_removals is 0
        def perform_update():
            env_indices_to_update = tf.squeeze(tf.cast(remove_indices, tf.int32), axis=1)
            pos_to_remove = tf.gather(new_pos, env_indices_to_update)
            scatter_indices = tf.stack([env_indices_to_update, pos_to_remove], axis=1)
            updates = tf.zeros(num_removals, dtype=tf.bool)
            return tf.tensor_scatter_nd_update(self.stock, scatter_indices, updates)

        maybe_updated_stock = tf.cond(tf.greater(num_removals, 0), true_fn=perform_update, false_fn=lambda: self.stock)
        self.stock.assign(maybe_updated_stock)

        is_valid_move = tf.logical_not(hit_shape_or_oob)
        next_pos = tf.where(is_valid_move, new_pos, self.pos)
        self.pos.assign(next_pos)

        self.steps.assign_add(tf.ones_like(self.steps))
        newly_done = (self.steps >= self.max_steps)
        self.done.assign(tf.logical_or(self.done, newly_done))

        next_obs = self._get_obs()
        return next_obs, tf.cast(reward, tf.float32), self.done

    @tf.function
    def _get_obs(self):
        G = self.G; N = self.N
        stock_grid = tf.reshape(self.stock, [N, G, G, G])
        shape_mask_grid = tf.reshape(self.shape_mask, [N, G, G, G])

        stock_float = tf.cast(stock_grid, tf.float32)
        shape_mask_float = tf.cast(shape_mask_grid, tf.float32)
        grid_obs = tf.stack([stock_float, shape_mask_float], axis=-1)

        g_tf = tf.constant(G, dtype=tf.int32)
        z = self.pos % g_tf
        y = (self.pos // g_tf) % g_tf
        x = self.pos // (g_tf * g_tf)

        # normalize coordinates to [0, 1] range
        g_minus_1_float = tf.cast(tf.maximum(1, G - 1), tf.float32) # avoid div by 0 issue
        x_norm = tf.cast(x, tf.float32) / g_minus_1_float
        y_norm = tf.cast(y, tf.float32) / g_minus_1_float
        z_norm = tf.cast(z, tf.float32) / g_minus_1_float


        coord_obs = tf.stack([x_norm, y_norm, z_norm], axis=-1)

        return (grid_obs, coord_obs)


# Replay buffers --> individual for worker / manager
# worker buffer: (S_w_grid, S_w_coord, G_w_index, A_w, R_w_total, S2_w_grid, S2_w_coord, D_w)
class WorkerReplayBuffer:
    def __init__(self, capacity=50000):
        self.cap = capacity
        self.buf = deque(maxlen=capacity)

    def add_batch(self, S_tuple, G_batch, A, R_total, S2_tuple, D):
        # S_tuple, S2_tuple are (grid_obs, coord_obs) [N, ...]
        # G_batch: goal index batch [N]
        # A, R_total, D are tensors [N]
        # store each (S_grid, S_coord, G_index, A, R, S2_grid, S2_coord, D) for each environment in the batch
        S_grid_batch, S_coord_batch = S_tuple
        S2_grid_batch, S2_coord_batch = S2_tuple
        N = tf.shape(A)[0].numpy() # Get batch size

        for i in range(N):
             self.buf.append((
                 S_grid_batch[i], S_coord_batch[i], # state S (single env)
                 G_batch[i],                       # goal G (single env)
                 A[i],                             # action A (single env)
                 R_total[i],                       # reward R (single env)
                 S2_grid_batch[i], S2_coord_batch[i], # next state S2 (single env)
                 D[i]                              # finished D (single env)
             ))

    def sample(self, batch_size=32):
        if len(self.buf) < batch_size:
            return None

        batch = random.sample(self.buf, batch_size)

        S_grid_list, S_coord_list, G_list, A_list, R_list, S2_grid_list, S2_coord_list, D_list = zip(*batch)

        return (
            tf.stack(S_grid_list, axis=0), tf.stack(S_coord_list, axis=0), # state S [B, ...]
            tf.stack(G_list, axis=0),                                     # goal G [B]
            tf.stack(A_list, axis=0),                                     # action A [B]
            tf.stack(R_list, axis=0),                                     # reward R [B]
            tf.stack(S2_grid_list, axis=0), tf.stack(S2_coord_list, axis=0), # next state S2 [B, ...]
            tf.stack(D_list, axis=0)                                      # done D [B]
        )

    def __len__(self):
        return len(self.buf)


# manager buffer: (S_m_grid, S_m_coord, G, R_m_accumulated_extrinsic, S'_m_grid, S'_m_coord, D_m)
class ManagerReplayBuffer:
    def __init__(self, capacity=5000): # manager buffer can be smaller
        self.cap = capacity
        self.buf = deque(maxlen=capacity)

    def add_batch(self, S_tuple, G, R_accumulated, S2_tuple, D):
        # S_tuple, S2_tuple are (grid_obs, coord_obs) at start/end of horizon [N, ...]
        # G is goal index [N]
        # R_accumulated is accumulated extrinsic reward [N]
        # D is done flag [N]

        S_grid_batch, S_coord_batch = S_tuple
        S2_grid_batch, S2_coord_batch = S2_tuple
        N = tf.shape(G)[0].numpy() # Get batch size

        for i in range(N):
             self.buf.append((
                 S_grid_batch[i], S_coord_batch[i], # state S (single env, start of horizon)
                 G[i],                             # goal G (single env)
                 R_accumulated[i],                 # accumulated reward R (single env)
                 S2_grid_batch[i], S2_coord_batch[i], # next state S2 (single env, end of horizon)
                 D[i]                              # done D (single env, episode ended)
             ))

    def sample(self, batch_size=32):
        if len(self.buf) < batch_size:
            return None

        batch = random.sample(self.buf, batch_size)

        S_grid_list, S_coord_list, G_list, R_list, S2_grid_list, S2_coord_list, D_list = zip(*batch)

        return (
            tf.stack(S_grid_list, axis=0), tf.stack(S_coord_list, axis=0), # state S [B, ...]
            tf.stack(G_list, axis=0),                                     # goal G [B]
            tf.stack(R_list, axis=0),                                     # reward R [B]
            tf.stack(S2_grid_list, axis=0), tf.stack(S2_coord_list, axis=0), # next state S2 [B, ...]
            tf.stack(D_list, axis=0)                                      # done D [B]
        )

    def __len__(self):
        return len(self.buf)


# factorized gaussian noise noisy layer -->
class NoisyDense(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, sigma0=0.5, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)
        self.sigma0 = sigma0

    def build(self, input_shape):
        in_features = input_shape[-1]
        out_features = self.units
        dtype = tf.float32

        # weight parameters (mean and standard deviation)
        sigma_init_val = self.sigma0 / math.sqrt(float(in_features))
        sigma_initializer = tf.constant_initializer(sigma_init_val)
        self.kernel_mean = self.add_weight(name="kernel_mean", shape=(in_features, out_features), initializer="he_uniform", trainable=True, dtype=dtype)
        self.kernel_sigma = self.add_weight(name="kernel_sigma", shape=(in_features, out_features), initializer=sigma_initializer, trainable=True, dtype=dtype)

        # bias parameters (mean and standard deviation)
        self.bias_mean = self.add_weight(name="bias_mean", shape=(out_features,), initializer="zeros", trainable=True, dtype=dtype)
        self.bias_sigma = self.add_weight(name="bias_sigma", shape=(out_features,), initializer=sigma_initializer, trainable=True, dtype=dtype)

        super().build(input_shape)

    def call(self, inputs, training=None):
        if training:
            # genearte noise for input and output dimensions
            noise_in = self._factorized_noise(tf.shape(inputs)[-1])
            noise_out = self._factorized_noise(self.units)

            # combine noise for weight matrix: outer product
            kernel_noise = tf.tensordot(tf.expand_dims(noise_in, -1), tf.expand_dims(noise_out, 0), axes=1)

            # noise for bias is just the output noise
            bias_noise = noise_out

            # apply noise: W = W_mu + W_sigma * noise_W, b = b_mu + b_sigma * noise_b
            kernel = self.kernel_mean + self.kernel_sigma * kernel_noise
            bias = self.bias_mean + self.bias_sigma * bias_noise
        else:
            # in inference mode use only the mean weights and biases
            kernel = self.kernel_mean
            bias = self.bias_mean

        output = tf.matmul(inputs, kernel) + bias

        if self.activation is not None:
            output = self.activation(output)
        return output

    # generate noise based on the gaussian noise formula.
    @tf.function
    def _factorized_noise(self, num_elements):
        noise = tf.random.normal(shape=[num_elements], dtype=tf.float32)
        return tf.sign(noise) * tf.sqrt(tf.abs(noise))

# Feudal agent
class FeudalAgentTF:
    def __init__(self, grid_shape, coord_shape, primitive_action_dim=6,
                 manager_goal_k=1, # discreet goals definition
                 subgoal_horizon=10, intrinsic_reward_beta=0.1,
                 manager_lr=1e-4, worker_lr=1e-4, gamma=0.99, tau=0.005):

        self.grid_shape = grid_shape
        self.coord_shape = coord_shape
        self.primitive_action_dim = primitive_action_dim
        self.gamma = gamma
        self.tau = tau
        self.subgoal_horizon = subgoal_horizon
        self.intrinsic_reward_beta = intrinsic_reward_beta

        # manager goal space
        self.manager_goal_k = manager_goal_k
        self.manager_goal_base = 2 * manager_goal_k + 1
        self.manager_goal_dim = self.manager_goal_base ** 3 # num of discrete goals

        print(f"Feudal Agent initialized:")
        print(f"  Subgoal Horizon (M): {self.subgoal_horizon}")
        print(f"  Intrinsic Reward Beta: {self.intrinsic_reward_beta}")
        print(f"  Manager Goal K: {self.manager_goal_k} (Discrete goals: {self.manager_goal_dim})")


        def build_manager_model(name="Manager_Model"):
            grid_input = tf.keras.layers.Input(shape=self.grid_shape, name="manager_grid_input")
            coord_input = tf.keras.layers.Input(shape=self.coord_shape, name="manager_coord_input")

            # CNN part for grid observations
            x_cnn = tf.keras.layers.Conv3D(filters=32, kernel_size=5, strides=2, activation='relu', padding='same', name="m_conv1")(grid_input)
            x_cnn = tf.keras.layers.Conv3D(filters=64, kernel_size=3, strides=2, activation='relu', padding='same', name="m_conv2")(x_cnn)
            x_cnn = tf.keras.layers.Conv3D(filters=64, kernel_size=3, strides=1, activation='relu', padding='same', name="m_conv3")(x_cnn)
            cnn_features = tf.keras.layers.Flatten(name="m_flatten")(x_cnn)

            # concat CNN features with coordinate observations
            concat_features = tf.keras.layers.Concatenate(name="m_concat")([cnn_features, coord_input])

            x = NoisyDense(256, activation='relu', name="manager_dense1")(concat_features) # Smaller dense for manager?
            outputs = NoisyDense(self.manager_goal_dim, activation='linear', name="manager_output")(x) # Q-values for each goal

            return tf.keras.Model(inputs=[grid_input, coord_input], outputs=outputs, name=name)

        def build_worker_model(name="Worker_Model"):
            grid_input = tf.keras.layers.Input(shape=self.grid_shape, name="worker_grid_input")
            coord_input = tf.keras.layers.Input(shape=self.coord_shape, name="worker_coord_input")
            # worker also takes the goal as input
            goal_input = tf.keras.layers.Input(shape=(self.manager_goal_dim,), name="worker_goal_input") # One-hot goal


            # CNN part for grid observations
            x_cnn = tf.keras.layers.Conv3D(filters=32, kernel_size=5, strides=2, activation='relu', padding='same', name="w_conv1")(grid_input)
            x_cnn = tf.keras.layers.Conv3D(filters=64, kernel_size=3, strides=2, activation='relu', padding='same', name="w_conv2")(x_cnn)
            x_cnn = tf.keras.layers.Conv3D(filters=64, kernel_size=3, strides=1, activation='relu', padding='same', name="w_conv3")(x_cnn)
            cnn_features = tf.keras.layers.Flatten(name="w_flatten")(x_cnn)

            # Concatenate CNN features with coordinate observations & one-hot goal
            concat_features = tf.keras.layers.Concatenate(name="w_concat")([cnn_features, coord_input, goal_input])


            x = NoisyDense(512, activation='relu', name="worker_dense1")(concat_features)
            outputs = NoisyDense(primitive_action_dim, activation='linear', name="worker_output")(x)

            return tf.keras.Model(inputs=[grid_input, coord_input, goal_input], outputs=outputs, name=name)

        self.manager_model = build_manager_model()
        self.manager_target = build_manager_model()
        self.manager_target.set_weights(self.manager_model.get_weights())

        self.worker_model = build_worker_model()
        self.worker_target = build_worker_model()
        self.worker_target.set_weights(self.worker_model.get_weights())

        self.manager_opt = tf.keras.optimizers.Adam(learning_rate=manager_lr)
        self.worker_opt = tf.keras.optimizers.Adam(learning_rate=worker_lr)

        self.manager_buffer = ManagerReplayBuffer()
        self.worker_buffer = WorkerReplayBuffer()

        logdir = f"runs/feudal_dqn_{datetime.datetime.now():%Y%m%d_%H%M%S}"
        self.writer = tf.summary.create_file_writer(logdir)
        print(f"TensorBoard log directory: {logdir}")

        self.manager_train_step_count = tf.Variable(0, dtype=tf.int64, trainable=False, name="manager_train_steps")
        self.worker_train_step_count = tf.Variable(0, dtype=tf.int64, trainable=False, name="worker_train_steps")

    # perform a single training step for the manager
    @tf.function
    def manager_train_step(self, S_grid, S_coord, G, R_accumulated, S2_grid, S2_coord, D):
        # double DQN compute Q values
        # get next goals from the current manager model for S2
        Q2_manager_online = self.manager_model([S2_grid, S2_coord], training=True)
        best_goals_next = tf.argmax(Q2_manager_online, axis=1, output_type=tf.int32)

        # get Q-values from the target manager model for S2
        Q2_manager_target = self.manager_target([S2_grid, S2_coord], training=True)

        # choose Q-value from the target network corresponding to the best goal selected by the online network
        batch_indices = tf.range(tf.shape(best_goals_next)[0], dtype=tf.int32)
        goal_indices = tf.stack([batch_indices, best_goals_next], axis=1)
        Q2_best_manager_target = tf.gather_nd(Q2_manager_target, goal_indices)

        # calculate the TD target: R_manager + gamma * Q_target(S', argmax_g Q_online(S', g)) * (1 - D_m)
        target_Q_manager = R_accumulated + self.gamma * Q2_best_manager_target * (1.0 - tf.cast(D, tf.float32))

        with tf.GradientTape() as tape:
            # predict q values
            Q_manager_online = self.manager_model([S_grid, S_coord], training=True)

            # select the Q-values
            goal_indices_taken = tf.stack([batch_indices, G], axis=1)
            Q_manager_online_taken = tf.gather_nd(Q_manager_online, goal_indices_taken)

            # calc loss (Mean Squared Error)
            loss = tf.keras.losses.MeanSquaredError()(target_Q_manager, Q_manager_online_taken)

        # compute and apply gradients
        grads = tape.gradient(loss, self.manager_model.trainable_variables)
        self.manager_opt.apply_gradients(zip(grads, self.manager_model.trainable_variables))

        for target_var, online_var in zip(self.manager_target.weights, self.manager_model.weights):
             target_var.assign(self.tau * online_var + (1.0 - self.tau) * target_var)

        return loss

    # perform a single training step for the Worker network
    @tf.function
    def worker_train_step(self, S_grid, S_coord, G_one_hot, A, R_total_worker, S2_grid, S2_coord, D):
        # double DQN update
        # get next actions from the *online* worker model for S2 + Goal
        Q2_worker_online = self.worker_model([S2_grid, S2_coord, G_one_hot], training=True)
        best_actions_next = tf.argmax(Q2_worker_online, axis=1, output_type=tf.int32)

        # get Q-values from the *target* worker model for S2 + Goal
        Q2_worker_target = self.worker_target([S2_grid, S2_coord, G_one_hot], training=True)

        # select the Q-value from the target network corresponding to the best action selected by the online network
        batch_indices = tf.range(tf.shape(best_actions_next)[0], dtype=tf.int32)
        action_indices = tf.stack([batch_indices, best_actions_next], axis=1)
        Q2_best_worker_target = tf.gather_nd(Q2_worker_target, action_indices)

        # calc target: R_total_worker + gamma * Q_target(S', G, argmax_a Q_online(S', G, a)) * (1 - D_w)
        target_Q_worker = R_total_worker + self.gamma * Q2_best_worker_target * (1.0 - tf.cast(D, tf.float32))

        with tf.GradientTape() as tape:
            Q_worker_online = self.worker_model([S_grid, S_coord, G_one_hot], training=True)

            action_indices_taken = tf.stack([batch_indices, A], axis=1)
            Q_worker_online_taken = tf.gather_nd(Q_worker_online, action_indices_taken)

            loss = tf.keras.losses.MeanSquaredError()(target_Q_worker, Q_worker_online_taken)

        grads = tape.gradient(loss, self.worker_model.trainable_variables)
        self.worker_opt.apply_gradients(zip(grads, self.worker_model.trainable_variables))

        for target_var, online_var in zip(self.worker_target.weights, self.worker_model.weights):
             target_var.assign(self.tau * online_var + (1.0 - self.tau) * target_var)

        return loss

    # selects a goal for a batch of states (manager)
    @tf.function
    def manager_act_batch(self, S_tuple, deterministic=False):
        q_values = self.manager_model(S_tuple, training=not deterministic)
        goals = tf.argmax(q_values, axis=1, output_type=tf.int32)
        return goals

    # worker selects a primitive action for a batch of states given goals
    @tf.function
    def worker_act_batch(self, S_tuple, G_batch, deterministic=False):
        S_grid, S_coord = S_tuple
        G_one_hot = tf.one_hot(G_batch, depth=self.manager_goal_dim, dtype=tf.float32)

        q_values = self.worker_model([S_grid, S_coord, G_one_hot], training=not deterministic)
        actions = tf.argmax(q_values, axis=1, output_type=tf.int32)

        return actions

    # add a batch of Manager transitions to the buffer
    def manager_remember_batch(self, S_tuple, G, R_accumulated, S2_tuple, D):
        if tf.shape(G)[0] > 0:
             self.manager_buffer.add_batch(S_tuple, G, R_accumulated, S2_tuple, D)

    # add a batch of Worker transitions (including goal) to the buffer
    def worker_remember_batch(self, S_tuple, G_batch, A, R_total, S2_tuple, D):
        if tf.shape(A)[0] > 0:
             self.worker_buffer.add_batch(S_tuple, G_batch, A, R_total, S2_tuple, D)

    # samples from worker buffer and performs a training step.
    def worker_learn(self, batch_size=32):
        if len(self.worker_buffer) < batch_size:
            return None

        sampled_data = self.worker_buffer.sample(batch_size)
        if sampled_data is None:
             return None

        S_grid_s, S_coord_s, G_s, A_s, R_total_s, S2_grid_s, S2_coord_s, D_s = sampled_data

        G_s_one_hot = tf.one_hot(tf.cast(G_s, tf.int32), depth=self.manager_goal_dim, dtype=tf.float32)

        loss = self.worker_train_step(S_grid_s, S_coord_s, G_s_one_hot, A_s, R_total_s, S2_grid_s, S2_coord_s, D_s)
        self.worker_train_step_count.assign_add(1)

        return loss.numpy()

    #   samples from manager buffer and performs a training step
    def manager_learn(self, batch_size=32):
        if len(self.manager_buffer) < batch_size:
            return None

        sampled_data = self.manager_buffer.sample(batch_size)
        if sampled_data is None:
             return None

        S_grid_s, S_coord_s, G_s, R_accumulated_s, S2_grid_s, S2_coord_s, D_s = sampled_data

        loss = self.manager_train_step(S_grid_s, S_coord_s, tf.cast(G_s, tf.int32), R_accumulated_s, S2_grid_s, S2_coord_s, D_s)
        self.manager_train_step_count.assign_add(1)

        return loss.numpy()

# evaluate trained feudal agent deterministically
def evaluate_agent_performance(agent, grid_size, max_steps, num_eval_episodes=10, render=True, render_env_index=0):
    print(f"\n**** Running Evaluation ({num_eval_episodes} episodes) ****")
    eval_start_time = time.time()

    eval_env = BatchedSculpt3DEnvTF(grid_size=grid_size, max_steps=max_steps, n_envs=num_eval_episodes)
    N_eval = num_eval_episodes

    initial_shape_mask_flat_gpu = eval_env.shape_mask[0]
    initial_shape_mask_flat_np = initial_shape_mask_flat_gpu.numpy()
    initial_carvable_mask_flat = ~initial_shape_mask_flat_np
    initial_carvable_count = np.sum(initial_carvable_mask_flat)
    print(f"  Initial number of carvable voxels: {initial_carvable_count}")
    if initial_carvable_count == 0: print("  ERROR: No carvable material defined.")

    all_ep_rewards = []
    all_ep_lengths = []
    all_ep_removed_counts = []
    all_ep_incorrect_removed_counts = []

    final_stock_variable_eval = eval_env.stock

    current_goals = tf.Variable(tf.zeros([N_eval], dtype=tf.int32), trainable=False, name="eval_goals") # Store goal index
    subgoal_steps_remaining = tf.Variable(tf.zeros([N_eval], dtype=tf.int32), trainable=False, name="eval_subgoal_steps")

    obs_tuple = eval_env.reset()
    done = eval_env.done

    ep_rewards = tf.Variable(tf.zeros([N_eval], dtype=tf.float32), trainable=False, name="eval_ep_rewards")
    ep_steps = tf.Variable(tf.zeros([N_eval], dtype=tf.int32), trainable=False, name="eval_ep_steps")

    manager_actions_initial = agent.manager_act_batch(obs_tuple, deterministic=True)
    current_goals.assign(manager_actions_initial)
    subgoal_steps_remaining.assign(tf.constant(agent.subgoal_horizon, dtype=tf.int32, shape=[N_eval]))

    # using tf.function for the evaluation step for performance
    @tf.function
    def evaluation_step(current_obs_tuple, current_done,
                         current_goals_var, subgoal_steps_remaining_var, ep_rewards_var, ep_steps_var,
                         eval_agent_subgoal_horizon, eval_env_step_fn, eval_agent_manager_act_fn, eval_agent_worker_act_fn):

         if tf.reduce_all(current_done): return current_obs_tuple, current_done # exit early if all envs are done

         obs_grid, obs_coord = current_obs_tuple

         end_of_horizon_mask = tf.equal(subgoal_steps_remaining_var.read_value(), 0)
         episode_done_mask = current_done
         manager_update_mask = tf.logical_or(end_of_horizon_mask, episode_done_mask)

         # find environments that need a manager update
         masked_indices = tf.where(manager_update_mask)[:, 0]
         num_masked = tf.shape(masked_indices)[0]


         # manager acts only if there is new goal
         if num_masked > 0:
             # get states for the masked subset
             manager_states_grid = tf.gather(obs_grid, masked_indices)
             manager_states_coord = tf.gather(obs_coord, masked_indices)
             manager_states = (manager_states_grid, manager_states_coord)

             # selects goals deterministically for these states
             new_goals_selected = eval_agent_manager_act_fn(manager_states, deterministic=True) # Use passed function

             # scatter the new goals back into the full batch variable
             current_goals_var.assign(tf.tensor_scatter_nd_update(current_goals_var.read_value(), tf.expand_dims(masked_indices, axis=1), new_goals_selected))

             # reset subgoal steps for these environments
             subgoal_steps_remaining_var.assign(tf.tensor_scatter_nd_update(
                 subgoal_steps_remaining_var.read_value(), tf.expand_dims(masked_indices, axis=1),
                 tf.fill([num_masked], eval_agent_subgoal_horizon))
             )

         # worker step
         # worker always acts for all environments using the current state and current goals
         A = eval_agent_worker_act_fn(current_obs_tuple, current_goals_var.read_value(), deterministic=True)

         # stepp the environment
         S2_tuple, R, next_done = eval_env_step_fn(A)


         # fix rewards and steps only for environments not yet done
         active_mask_current = ~current_done
         ep_rewards_var.assign_add(R * tf.cast(active_mask_current, tf.float32))
         ep_steps_var.assign_add(tf.cast(active_mask_current, tf.int32))

         # decrement subgoal steps for active environments
         active_mask_next = ~next_done
         subgoal_steps_remaining_var.assign(tf.where(active_mask_next, subgoal_steps_remaining_var.read_value() - 1, subgoal_steps_remaining_var.read_value()))

         return S2_tuple, next_done

    for _ in range(max_steps):
         obs_tuple, done = evaluation_step(
             obs_tuple, done,
             current_goals, subgoal_steps_remaining, ep_rewards, ep_steps,
             agent.subgoal_horizon, eval_env.step, agent.manager_act_batch, agent.worker_act_batch
         )
         if tf.reduce_all(done): break

    final_stock_batch_np = final_stock_variable_eval.numpy() # get final stock state after carving is completed
    batch_rewards = ep_rewards.numpy()
    batch_lengths = ep_steps.numpy()

    all_ep_rewards.extend(batch_rewards.tolist())
    all_ep_lengths.extend(batch_lengths.tolist())

    # calculate removed/incorrect counts per episode
    for i in range(N_eval):
        final_stock_flat_np = final_stock_batch_np[i]

        # correctt removed: carvable and now carved by model
        removed_mask = initial_carvable_mask_flat & (~final_stock_flat_np)
        removed_count = np.sum(removed_mask)
        all_ep_removed_counts.append(removed_count)

        # incorrectly removed:  part of shape and incirrectly cut by model
        incorrectly_removed_mask = initial_shape_mask_flat_np & (~final_stock_flat_np)
        incorrectly_removed_count = np.sum(incorrectly_removed_mask)
        all_ep_incorrect_removed_counts.append(incorrectly_removed_count)

    avg_reward = np.mean(all_ep_rewards); std_reward = np.std(all_ep_rewards)
    avg_length = np.mean(all_ep_lengths)
    avg_removed_count = np.mean(all_ep_removed_counts)
    avg_incorrect_removed = np.mean(all_ep_incorrect_removed_counts)

    if initial_carvable_count > 0:
        removal_percentages = [(c / initial_carvable_count) * 100.0 for c in all_ep_removed_counts]
        avg_removal_percentage = np.mean(removal_percentages)
        std_removal_percentage = np.std(removal_percentages)
    else:
        avg_removal_percentage = 0.0
        std_removal_percentage = 0.0

    print(f"\n*** Evaluation Results ***")
    print(f"  Avg Reward : {avg_reward:.2f} (+/- {std_reward:.2f})")
    print(f"  Avg Length : {avg_length:.1f}")
    print(f"  Avg Removed: {avg_removed_count:.1f} / {initial_carvable_count} ({avg_removal_percentage:.2f}% +/- {std_removal_percentage:.2f}%)")
    print(f"  Avg Incorrect: {avg_incorrect_removed:.1f}")
    eval_duration = time.time() - eval_start_time
    print(f"  Evaluation Duration: {eval_duration:.2f}s")

    # rendering of final state after cuts made
    if render:
        print(f"\n***Rendering final state for Eval Env Index: {render_env_index} ***")
        if render_env_index < 0 or render_env_index >= N_eval:
            print(f"Error: render_env_index ({render_env_index}) out of bounds for {N_eval} eval envs.")
        elif N_eval > 0:
            try:
                final_stock_flat_np = final_stock_batch_np[render_env_index]
                shape_mask_flat_np = initial_shape_mask_flat_np

                G = grid_size
                final_stock_3d = final_stock_flat_np.reshape((G, G, G))
                shape_mask_3d = shape_mask_flat_np.reshape((G, G, G))

                shape_to_plot = shape_mask_3d # target
                initial_carvable_mask_render = ~shape_mask_3d # initially carvable
                removed_mask_render = initial_carvable_mask_render & (~final_stock_3d) # correctly removed
                incorrectly_removed_mask_render = shape_mask_3d & (~final_stock_3d) # incorrectly removed

                shape_np = shape_to_plot

                fig = plt.figure(figsize=(9, 7)); ax = fig.add_subplot(111, projection='3d')
                ax.set_facecolor('whitesmoke')
                x_vox, y_vox, z_vox = np.indices(np.array(shape_to_plot.shape) + 1)

                ax.voxels(x_vox, y_vox, z_vox, shape_to_plot, facecolors='blue', alpha=0.1, edgecolor=None)

                ax.voxels(x_vox, y_vox, z_vox, removed_mask_render, facecolors='red', alpha=0.6, edgecolor=None)

                if np.sum(incorrectly_removed_mask_render) > 0:
                     ax.voxels(x_vox, y_vox, z_vox, incorrectly_removed_mask_render, facecolors='yellow', alpha=0.7, edgecolor='orange', label='Incorrect Removal')
                     ax.legend()

                rendered_env_reward = all_ep_rewards[render_env_index]
                rendered_env_removed = all_ep_removed_counts[render_env_index]
                rendered_env_perc = (rendered_env_removed / initial_carvable_count) * 100.0 if initial_carvable_count > 0 else 0.0
                ax.set_title(f"Eval Render Env #{render_env_index} (R={rendered_env_reward:.1f}, Removed={rendered_env_perc:.1f}%)")
                ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
                ax.set_xlim(0, G); ax.set_ylim(0, G); ax.set_zlim(0, G); ax.set_aspect('auto') # Use 'auto' or 'equal'
                plt.tight_layout()

                render_dir = "renders_eval_feudal"
                os.makedirs(render_dir, exist_ok=True)
                save_path = os.path.join(render_dir, f"eval_render_env_{render_env_index}_final.png")
                plt.savefig(save_path); print(f"Saved evaluation render to {save_path}"); plt.close(fig)
            except Exception as e:
                print(f"Error during rendering: {e}")
                import traceback; traceback.print_exc()

    return {
        "avg_reward": avg_reward,
        "std_reward": std_reward,
        "avg_length": avg_length,
        "avg_removed_count": avg_removed_count,
        "avg_incorrect_removed": avg_incorrect_removed,
        "avg_removal_percentage": avg_removal_percentage,
        "std_removal_percentage": std_removal_percentage,
        "initial_carvable_count": initial_carvable_count
    }


# Training loop
def train_gpu_batched_feudal(
    grid_size=16, max_steps=300, n_envs=32, episodes=10000,
    worker_buffer_capacity=100000, manager_buffer_capacity=10000,
    worker_learn_batch_size=32, manager_learn_batch_size=32,
    worker_learn_freq=4, manager_learn_freq=100,
    gamma=0.99, manager_lr=1e-4, worker_lr=1e-4, tau=0.005,
    subgoal_horizon=10, intrinsic_reward_beta=0.1, manager_goal_k=1,
    log_every=50, evaluate_every=100, num_eval_episodes_periodic=10,
    render_intermediate_eval=False, save_every_episodes=500,
    checkpoint_dir="checkpoints_feudal"
    ):

    print(f"*** Training Feudal Agent ***")
    print(f"Params: Grid={grid_size}, N_Envs={n_envs}, MaxSteps={max_steps}, Episodes={episodes}")
    print(f"Subgoal Horizon: {subgoal_horizon}, Intrinsic Beta: {intrinsic_reward_beta}, Goal K: {manager_goal_k}")
    print(f"Worker Learn (B): {worker_learn_batch_size}, Freq: {worker_learn_freq} env steps")
    print(f"Manager Learn (B): {manager_learn_batch_size}, Freq: {manager_learn_freq} worker learn steps")
    print(f"Periodic Evaluation every {evaluate_every} episodes ({num_eval_episodes_periodic} eps each).")
    print(f"Periodic Weight Saving every {save_every_episodes} episodes to '{checkpoint_dir}'.")
    print(f"Memory Warning: Ensure sufficient CPU RAM and GPU VRAM.")


    env = BatchedSculpt3DEnvTF(grid_size, max_steps, n_envs)
    agent = FeudalAgentTF(grid_shape=env.grid_obs_shape, coord_shape=env.coord_obs_shape,
                          primitive_action_dim=6, manager_goal_k=manager_goal_k,
                          subgoal_horizon=subgoal_horizon, intrinsic_reward_beta=intrinsic_reward_beta,
                          manager_lr=manager_lr, worker_lr=worker_lr, gamma=gamma, tau=tau)

    agent.worker_buffer.cap = worker_buffer_capacity
    agent.manager_buffer.cap = manager_buffer_capacity


    total_env_steps_taken = 0
    episode_rewards_history = []
    episode_lengths_history = []
    worker_losses_history = deque(maxlen=log_every)
    manager_losses_history = deque(maxlen=log_every)
    start_time = time.time()

    eval_episodes_list = []
    eval_avg_rewards_list = []
    eval_avg_removal_perc_list = []

    current_goals = tf.Variable(tf.zeros([n_envs], dtype=tf.int32), trainable=False, name="current_goals")
    subgoal_steps_remaining = tf.Variable(tf.zeros([n_envs], dtype=tf.int32), trainable=False, name="subgoal_steps_remaining")
    current_subgoal_start_pos = tf.Variable(tf.zeros([n_envs], dtype=tf.int32), trainable=False, name="subgoal_start_pos")
    current_subgoal_start_obs_grid = tf.Variable(tf.zeros([n_envs, *env.grid_obs_shape], dtype=tf.float32), trainable=False, name="subgoal_start_obs_grid")
    current_subgoal_start_obs_coord = tf.Variable(tf.zeros([n_envs, *env.coord_obs_shape], dtype=tf.float32), trainable=False, name="subgoal_start_obs_coord")
    accumulated_extrinsic_rewards_current_horizon = tf.Variable(tf.zeros([n_envs], dtype=tf.float32), trainable=False, name="acc_ext_rewards")
    has_valid_prev_manager_transition_data = tf.Variable(tf.zeros([n_envs], dtype=tf.bool), trainable=False, name="has_prev_m_trans")

    ep_rewards = tf.Variable(tf.zeros([n_envs], dtype=tf.float32), trainable=False, name="episode_rewards")
    ep_steps = tf.Variable(tf.zeros([n_envs], dtype=tf.int32), trainable=False, name="episode_steps")

    @tf.function
    def training_step(current_obs_tuple, current_done, env_pos,
                      current_goals_var, subgoal_steps_remaining_var, current_subgoal_start_pos_var,
                      current_subgoal_start_obs_grid_var, current_subgoal_start_obs_coord_var,
                      accumulated_extrinsic_rewards_current_horizon_var,
                      has_valid_prev_manager_transition_data_var,
                      ep_rewards_var, ep_steps_var,
                      env_G, agent_subgoal_horizon, agent_manager_goal_k, agent_intrinsic_reward_beta,
                      env_step_fn, agent_manager_act_fn, agent_worker_act_fn):

        if tf.reduce_all(current_done):
            # return empty transition data if all envs are done
            empty_grid = tf.zeros([0, *env.grid_obs_shape], dtype=tf.float32)
            empty_coord = tf.zeros([0, *env.coord_obs_shape], dtype=tf.float32)
            empty_scalar_int = tf.zeros([0], dtype=tf.int32)
            empty_scalar_float = tf.zeros([0], dtype=tf.float32)
            empty_scalar_bool = tf.zeros([0], dtype=tf.bool)

            manager_transition_data = ((empty_grid, empty_coord), empty_scalar_int, empty_scalar_float, (empty_grid, empty_coord), empty_scalar_bool)
            worker_transition_data = ((empty_grid, empty_coord), empty_scalar_int, empty_scalar_int, empty_scalar_float, (empty_grid, empty_coord), empty_scalar_bool)
            return current_obs_tuple, current_done, manager_transition_data, worker_transition_data


        obs_grid, obs_coord = current_obs_tuple

        # mask for end of horizon goal
        end_of_horizon_mask = tf.equal(subgoal_steps_remaining_var.read_value(), 0)
        episode_done_mask = current_done # The 'done' flag from the previous step
        manager_update_mask = tf.logical_or(end_of_horizon_mask, episode_done_mask)

        # find environments that need a manager update
        masked_indices = tf.where(manager_update_mask)[:, 0]
        num_masked = tf.shape(masked_indices)[0]

        # empty transition data
        empty_grid = tf.zeros([0, *env.grid_obs_shape], dtype=tf.float32)
        empty_coord = tf.zeros([0, *env.coord_obs_shape], dtype=tf.float32)
        empty_scalar_int = tf.zeros([0], dtype=tf.int32)
        empty_scalar_float = tf.zeros([0], dtype=tf.float32)
        empty_scalar_bool = tf.zeros([0], dtype=tf.bool)

        manager_transition_data = (
             (empty_grid, empty_coord), empty_scalar_int, empty_scalar_float, (empty_grid, empty_coord), empty_scalar_bool
        )

        valid_transition_mask_in_masked = tf.gather(has_valid_prev_manager_transition_data_var.read_value(), masked_indices)

        if tf.reduce_any(valid_transition_mask_in_masked):
             valid_masked_indices = tf.boolean_mask(masked_indices, valid_transition_mask_in_masked)

             # collect transition data
             S_m_grid_batch = tf.gather(current_subgoal_start_obs_grid_var.read_value(), valid_masked_indices)
             S_m_coord_batch = tf.gather(current_subgoal_start_obs_coord_var.read_value(), valid_masked_indices)
             G_m_batch = tf.gather(current_goals_var.read_value(), valid_masked_indices)
             R_m_batch = tf.gather(accumulated_extrinsic_rewards_current_horizon_var.read_value(), valid_masked_indices)
             S2_m_grid_batch = tf.gather(obs_grid, valid_masked_indices)
             S2_m_coord_batch = tf.gather(obs_coord, valid_masked_indices)
             D_m_batch = tf.gather(current_done, valid_masked_indices)

             manager_transition_data = (
                 (S_m_grid_batch, S_m_coord_batch), G_m_batch, R_m_batch, (S2_m_grid_batch, S2_m_coord_batch), D_m_batch
             )

        # manager act, set new goal
        if num_masked > 0:
             # get states for the masked subset that need a new goal
             manager_act_states_grid = tf.gather(obs_grid, masked_indices)
             manager_act_states_coord = tf.gather(obs_coord, masked_indices)
             manager_act_states = (manager_act_states_grid, manager_act_states_coord)

             # selects new goals
             new_goals_selected = agent_manager_act_fn(manager_act_states, deterministic=False) # Use noise during training

             # update feudal state variables for the environments that needed an update
             scatter_indices = tf.expand_dims(masked_indices, axis=1)

             current_goals_var.assign(tf.tensor_scatter_nd_update(current_goals_var.read_value(), scatter_indices, new_goals_selected))
             current_subgoal_start_pos_var.assign(tf.tensor_scatter_nd_update(current_subgoal_start_pos_var.read_value(), scatter_indices, tf.gather(env_pos, masked_indices))) # Start pos is current pos
             current_subgoal_start_obs_grid_var.assign(tf.tensor_scatter_nd_update(current_subgoal_start_obs_grid_var.read_value(), scatter_indices, tf.gather(obs_grid, masked_indices)))
             current_subgoal_start_obs_coord_var.assign(tf.tensor_scatter_nd_update(current_subgoal_start_obs_coord_var.read_value(), scatter_indices, tf.gather(obs_coord, masked_indices)))
             accumulated_extrinsic_rewards_current_horizon_var.assign(tf.tensor_scatter_nd_update(accumulated_extrinsic_rewards_current_horizon_var.read_value(), scatter_indices, tf.fill([num_masked], 0.0))) # Reset accumulated rewards
             subgoal_steps_remaining_var.assign(tf.tensor_scatter_nd_update(subgoal_steps_remaining_var.read_value(), scatter_indices, tf.fill([num_masked], agent_subgoal_horizon))) # Reset step counter
             has_valid_prev_manager_transition_data_var.assign(tf.tensor_scatter_nd_update(has_valid_prev_manager_transition_data_var.read_value(), scatter_indices, tf.fill([num_masked], True))) # Now we have valid prev data

        pos_t = env_pos
        A = agent_worker_act_fn(current_obs_tuple, current_goals_var.read_value(), deterministic=False)
        S2_tuple, R_extrinsic, next_done = env_step_fn(A)

        R_intrinsic = calculate_intrinsic_reward_batch(pos_t, env.pos, current_subgoal_start_pos_var.read_value(), current_goals_var.read_value(), env_G, agent_manager_goal_k)

        R_total_worker = R_extrinsic + agent_intrinsic_reward_beta * R_intrinsic

        worker_transition_data = (current_obs_tuple, current_goals_var.read_value(), A, R_total_worker, S2_tuple, next_done)

        active_mask_next = ~next_done
        accumulated_extrinsic_rewards_current_horizon_var.assign(tf.where(active_mask_next, accumulated_extrinsic_rewards_current_horizon_var.read_value() + R_extrinsic, accumulated_extrinsic_rewards_current_horizon_var.read_value()))

        # update episode stats for active environments
        active_mask_current = ~current_done
        ep_rewards_var.assign_add(R_extrinsic * tf.cast(active_mask_current, tf.float32))
        ep_steps_var.assign_add(tf.cast(active_mask_current, tf.int32))

        # decrement subgoal steps for active environments
        subgoal_steps_remaining_var.assign(tf.where(active_mask_next, subgoal_steps_remaining_var.read_value() - 1, subgoal_steps_remaining_var.read_value()))

        return S2_tuple, next_done, manager_transition_data, worker_transition_data

    # main training loop
    for ep in range(1, episodes + 1):
        obs_tuple = env.reset() # initail stage for batch
        done = env.done

        ep_rewards.assign(tf.zeros([n_envs], dtype=tf.float32))
        ep_steps.assign(tf.zeros([n_envs], dtype=tf.int32))

        subgoal_steps_remaining.assign(tf.zeros([n_envs], dtype=tf.int32))
        has_valid_prev_manager_transition_data.assign(tf.zeros([n_envs], dtype=tf.bool))

        # episode step loop
        for current_ep_step in range(max_steps):
            # pass in the current state and done flags, and the tf.Variables that track feudal state
            obs_tuple, done, manager_transition_data, worker_transition_data = training_step(
                obs_tuple, done, env.pos, # Pass env.pos Variable directly
                current_goals, subgoal_steps_remaining, current_subgoal_start_pos,
                current_subgoal_start_obs_grid, current_subgoal_start_obs_coord,
                accumulated_extrinsic_rewards_current_horizon,
                has_valid_prev_manager_transition_data,
                ep_rewards, ep_steps,
                env.G, agent.subgoal_horizon, agent.manager_goal_k, agent.intrinsic_reward_beta,
                env.step, agent.manager_act_batch, agent.worker_act_batch
            )

            if tf.shape(manager_transition_data[1])[0] > 0:
                 agent.manager_remember_batch(*manager_transition_data)

            agent.worker_remember_batch(*worker_transition_data)

            total_env_steps_taken += n_envs # each env contributes 1 step

            # learn Worker periodically based on total environment steps
            if total_env_steps_taken > 0 and total_env_steps_taken % (worker_learn_freq * n_envs) == 0:
                worker_loss_val = agent.worker_learn(worker_learn_batch_size)
                if worker_loss_val is not None:
                    worker_losses_history.append(worker_loss_val)

            # learn Manager periodically based on Worker learn steps count
            # manager learns less often
            if agent.worker_train_step_count.numpy() > 0 and agent.worker_train_step_count.numpy() % manager_learn_freq == 0:
                 manager_loss_val = agent.manager_learn(manager_learn_batch_size)
                 if manager_loss_val is not None:
                     manager_losses_history.append(manager_loss_val)

            if tf.reduce_all(done): break

        # calculate and store average batch stats for the completed episode
        avg_reward_batch = tf.reduce_mean(ep_rewards).numpy()
        avg_steps_batch = tf.reduce_mean(tf.cast(ep_steps, tf.float32)).numpy()
        episode_rewards_history.append(avg_reward_batch)
        episode_lengths_history.append(avg_steps_batch)


        # log it alll
        if ep % log_every == 0 or ep == 1:
            elapsed_time = time.time() - start_time
            avg_r = np.mean(episode_rewards_history[-log_every:]) if len(episode_rewards_history) >= log_every else np.mean(episode_rewards_history)
            avg_l = np.mean(episode_lengths_history[-log_every:]) if len(episode_lengths_history) >= log_every else np.mean(episode_lengths_history)
            avg_worker_loss = np.mean(worker_losses_history) if worker_losses_history else 0.0 # Use deque rolling avg
            avg_manager_loss = np.mean(manager_losses_history) if manager_losses_history else 0.0

            print(f"Ep {ep}/{episodes} | Avg R (last {log_every}): {avg_r:.2f} | Avg Len: {avg_l:.1f} | EnvSteps: {total_env_steps_taken} | WorkerSteps: {agent.worker_train_step_count.numpy()} | ManagerSteps: {agent.manager_train_step_count.numpy()} | Time: {elapsed_time:.1f}s")
            print(f"  Losses: Worker {avg_worker_loss:.4f}, Manager {avg_manager_loss:.4f}")

            with agent.writer.as_default(step=ep):
                tf.summary.scalar("Episode/AvgReward_Roll", avg_r)
                tf.summary.scalar("Episode/AvgLength_Roll", avg_l)
                tf.summary.scalar("System/TotalEnvSteps", total_env_steps_taken)
            with agent.writer.as_default(step=agent.worker_train_step_count.numpy()):
                tf.summary.scalar("Train/WorkerLoss_Roll", avg_worker_loss)
                tf.summary.scalar("Train/ManagerLoss_Roll", avg_manager_loss)

            try:
                 render_env_index = 0
                 stock_np = env.stock.numpy()[render_env_index].reshape((grid_size, grid_size, grid_size))
                 shape_np = env.shape_mask.numpy()[render_env_index].reshape((grid_size, grid_size, grid_size))

                 removed_mask = (~shape_np) & (~stock_np)
                 incorrect_mask = shape_np & (~stock_np)

                 fig = plt.figure(figsize=(6, 5))
                 ax = fig.add_subplot(111, projection='3d')
                 x_vox, y_vox, z_vox = np.indices(np.array(stock_np.shape) + 1)

                 ax.voxels(x_vox, y_vox, z_vox, shape_np, facecolors='blue', alpha=0.1)
                 ax.voxels(x_vox, y_vox, z_vox, removed_mask, facecolors='red', alpha=0.6)
                 if np.sum(incorrect_mask) > 0:
                     ax.voxels(x_vox, y_vox, z_vox, incorrect_mask, facecolors='yellow', alpha=0.7)

                 ax.set_title(f"Ep {ep} - Render Env #{render_env_index}")
                 ax.set_axis_off()
                 fig.tight_layout()

                 buf = io.BytesIO()
                 plt.savefig(buf, format='png')
                 buf.seek(0)

                 image_tensor = tf.image.decode_png(buf.getvalue(), channels=4)
                 image_tensor = tf.expand_dims(image_tensor, 0)
                 buf.close()
                 plt.close(fig)

                 with agent.writer.as_default(step=ep):
                     tf.summary.image("EnvRender/Env0_Train", image_tensor)
            except Exception as e:
                 print(f"Render log error at ep {ep}: {e}")
                 import traceback; traceback.print_exc()

        # save model periodically to checkpoint dir
        if save_every_episodes > 0 and ep % save_every_episodes == 0 and ep > 0:
            try:
                os.makedirs(checkpoint_dir, exist_ok=True)
                manager_save_path = os.path.join(checkpoint_dir, f"manager_ep{ep}_g{grid_size}.weights.h5")
                worker_save_path = os.path.join(checkpoint_dir, f"worker_ep{ep}_g{grid_size}.weights.h5")
                agent.manager_model.save_weights(manager_save_path)
                agent.worker_model.save_weights(worker_save_path)
                print(f"\n--- Saved model weights at episode {ep} to {checkpoint_dir} ---")
            except Exception as e:
                print(f"\n--- Error saving weights at episode {ep}: {e} ---")
                import traceback; traceback.print_exc()

        # evalulate model periodically
        if evaluate_every > 0 and ep % evaluate_every == 0:
            eval_stats = evaluate_agent_performance(
                agent=agent,
                grid_size=grid_size,
                max_steps=max_steps,
                num_eval_episodes=num_eval_episodes_periodic,
                render=render_intermediate_eval,
                render_env_index=0
            )
            # store res for final trend plot
            if eval_stats:
                eval_episodes_list.append(ep)
                eval_avg_rewards_list.append(eval_stats["avg_reward"])
                eval_avg_removal_perc_list.append(eval_stats["avg_removal_percentage"])

                # log evaluation
                with agent.writer.as_default(step=ep):
                    tf.summary.scalar("Evaluate/AvgReward", eval_stats["avg_reward"])
                    tf.summary.scalar("Evaluate/AvgRemovalPercentage", eval_stats["avg_removal_percentage"])
                    tf.summary.scalar("Evaluate/AvgIncorrectRemoved", eval_stats["avg_incorrect_removed"])

            print("*" * 60)

    agent.writer.close()
    total_training_time = time.time() - start_time
    print(f"\nTraining finished. Total env steps: {total_env_steps_taken}, Total Time: {total_training_time:.2f}s")
    print(f"Worker train steps: {agent.worker_train_step_count.numpy()}, Manager train steps: {agent.manager_train_step_count.numpy()}")

    if eval_episodes_list:
        print("\n--- Plotting Evaluation Trend ---")
        try:
            fig, ax1 = plt.subplots(figsize=(12, 6))

            color = 'tab:red'
            ax1.set_xlabel('Training Episode')
            ax1.set_ylabel('Avg Carvable Material Removed (%)', color=color)
            ax1.plot(eval_episodes_list, eval_avg_removal_perc_list, color=color, marker='o', linestyle='-', label='Removal %')
            ax1.tick_params(axis='y', labelcolor=color)
            ax1.grid(True, axis='y', linestyle=':')

            ax2 = ax1.twinx()
            color = 'tab:blue'
            ax2.set_ylabel('Avg Evaluation Reward', color=color)
            ax2.plot(eval_episodes_list, eval_avg_rewards_list, color=color, marker='x', linestyle='--', label='Avg Reward')
            ax2.tick_params(axis='y', labelcolor=color)


            fig.suptitle('Feudal Agent Evaluation Performance During Training')
            fig.tight_layout(rect=[0, 0.03, 1, 0.95])

            plot_dir = "plots_feudal"
            os.makedirs(plot_dir, exist_ok=True)
            plot_path = os.path.join(plot_dir, f"evaluation_trend_g{grid_size}_n{n_envs}.png")
            plt.savefig(plot_path)
            print(f"Saved evaluation trend plot to {plot_path}")
            plt.close(fig)

        except Exception as e:
            print(f"Error plotting evaluation trend: {e}")

    return agent


# main execution block
if __name__ == "__main__":
    GRID_SIZE_RUN = 8 # resolution of grid (8x8) to save on compute
    N_ENVS_RUN = 16
    MAX_STEPS_RUN = 450 # max steps per ep
    EPISODES_RUN = 10000 # Total training episodes

    WORKER_BUFFER_CAP_RUN = 100000 # replay buff capacity
    MANAGER_BUFFER_CAP_RUN = 10000  # manager buff smaller

    WORKER_LEARN_BATCH_RUN = 32
    MANAGER_LEARN_BATCH_RUN = 32

    WORKER_LEARN_FREQ_RUN = 4    # worker learns approx every 4 env steps
    MANAGER_LEARN_FREQ_RUN = 100 # manager learns approx every 100 worker learn steps

    MANAGER_LEARNING_RATE = 1e-4
    WORKER_LEARNING_RATE = 1e-4
    GAMMA = 0.99       # discount factor
    TAU = 0.005        # targett network update rate

    SUBGOAL_HORIZON_RUN = 10 # manager sets a new goal every 10 steps
    INTRINSIC_REWARD_BETA_RUN = 0.1 # weight of reward for worker
    MANAGER_GOAL_K_RUN = 1 # relative displacement

    LOG_EVERY_RUN = 50
    EVAL_FREQ_RUN = 100      # eval every N training episodes
    NUM_EVAL_EPISODES_RUN = 10 # num of episodes per evaluation run
    RENDER_INTERMEDIATE_EVAL_RUN = False # rend evaluation plots during training?
    SAVE_FREQ_RUN = 1000      # save weights every N episodes
    CHECKPOINT_DIR_RUN = "checkpoints_feudal" # dir for saved weights


    print(f"Starting run with Feudal Agent")
    print(f"Params: Grid={GRID_SIZE_RUN}, N_Envs={N_ENVS_RUN}")
    print(f"Worker Learn (B): {WORKER_LEARN_BATCH_RUN}, Freq: {WORKER_LEARN_FREQ_RUN} env steps")
    print(f"Manager Learn (B): {MANAGER_LEARN_BATCH_RUN}, Freq: {MANAGER_LEARN_FREQ_RUN} worker learn steps")
    print(f"Feudal Params: Horizon={SUBGOAL_HORIZON_RUN}, Beta={INTRINSIC_REWARD_BETA_RUN}, GoalK={MANAGER_GOAL_K_RUN}")


    # train the agent with periodic evaluation and saving
    trained_agent = train_gpu_batched_feudal(
        grid_size=GRID_SIZE_RUN,
        max_steps=MAX_STEPS_RUN,
        n_envs=N_ENVS_RUN,
        episodes=EPISODES_RUN,
        worker_buffer_capacity=WORKER_BUFFER_CAP_RUN,
        manager_buffer_capacity=MANAGER_BUFFER_CAP_RUN,
        worker_learn_batch_size=WORKER_LEARN_BATCH_RUN,
        manager_learn_batch_size=MANAGER_LEARN_BATCH_RUN,
        worker_learn_freq=WORKER_LEARN_FREQ_RUN,
        manager_learn_freq=MANAGER_LEARN_FREQ_RUN,
        gamma=GAMMA,
        manager_lr=MANAGER_LEARNING_RATE,
        worker_lr=WORKER_LEARNING_RATE,
        tau=TAU,
        subgoal_horizon=SUBGOAL_HORIZON_RUN,
        intrinsic_reward_beta=INTRINSIC_REWARD_BETA_RUN,
        manager_goal_k=MANAGER_GOAL_K_RUN,
        log_every=LOG_EVERY_RUN,
        evaluate_every=EVAL_FREQ_RUN,
        num_eval_episodes_periodic=NUM_EVAL_EPISODES_RUN,
        render_intermediate_eval=RENDER_INTERMEDIATE_EVAL_RUN,
        save_every_episodes=SAVE_FREQ_RUN,
        checkpoint_dir=CHECKPOINT_DIR_RUN
    )

    # run eval
    print("\n" + "="*70)
    print("      RUNNING FINAL EVALUATION ON TRAINED FEUDAL AGENT")
    print("="*70)
    if trained_agent:
        evaluate_agent_performance(
            agent=trained_agent,
            grid_size=GRID_SIZE_RUN,
            max_steps=MAX_STEPS_RUN,
            num_eval_episodes=50,
            render=True,
            render_env_index=0
        )


    # saved trained model final weights!
    if trained_agent:
        final_manager_save_path = os.path.join(CHECKPOINT_DIR_RUN, f"FINAL_manager_ep{EPISODES_RUN}_g{GRID_SIZE_RUN}.weights.h5")
        final_worker_save_path = os.path.join(CHECKPOINT_DIR_RUN, f"FINAL_worker_ep{EPISODES_RUN}_g{GRID_SIZE_RUN}.weights.h5")
        try:
            os.makedirs(CHECKPOINT_DIR_RUN, exist_ok=True)
            trained_agent.manager_model.save_weights(final_manager_save_path)
            trained_agent.worker_model.save_weights(final_worker_save_path)
            print(f"\nFinal manager weights saved to {final_manager_save_path}")
            print(f"Final worker weights saved to {final_worker_save_path}")
        except Exception as e:
            print(f"\nError saving final model weights: {e}")
            import traceback; traceback.print_exc()

No GPU detected by TensorFlow.
Starting run with Feudal Agent
Params: Grid=8, N_Envs=16
Worker Learn (B): 32, Freq: 4 env steps
Manager Learn (B): 32, Freq: 100 worker learn steps
Feudal Params: Horizon=10, Beta=0.1, GoalK=1
--- Training Feudal Agent ---
Params: Grid=8, N_Envs=16, MaxSteps=450, Episodes=10000
Subgoal Horizon: 10, Intrinsic Beta: 0.1, Goal K: 1
Worker Learn (B): 32, Freq: 4 env steps
Manager Learn (B): 32, Freq: 100 worker learn steps
Periodic Evaluation every 100 episodes (10 eps each).
Periodic Weight Saving every 1000 episodes to 'checkpoints_feudal'.
Memory Warning: Ensure sufficient CPU RAM and GPU VRAM.
Feudal Agent initialized:
  Subgoal Horizon (M): 10
  Intrinsic Reward Beta: 0.1
  Manager Goal K: 1 (Discrete goals: 27)
TensorBoard log directory: runs/feudal_dqn_20250430_145320


KeyboardInterrupt: 

In [6]:
# --- Evaluation Cell for Saved Checkpoints ---
print("\n" + "="*70)
print("      EVALUATING SAVED FEUDAL AGENT CHECKPOINT")
print("="*70)

# MUST must match the training parameters used when saving the checkpoint
EVAL_GRID_SIZE = 8
EVAL_MAX_STEPS = 450
EVAL_N_EVAL_EPISODES = 50
EVAL_CHECKPOINT_DIR = "checkpoints_feudal"
EVAL_EPISODE_NUM = 10

# MUST match trained agent params
EVAL_MANAGER_GOAL_K = 1
EVAL_SUBGOAL_HORIZON = 10
EVAL_INTRINSIC_REWARD_BETA = 0.1
EVAL_MANAGER_LR = 1e-4
EVAL_WORKER_LR = 1e-4
EVAL_GAMMA = 0.99
EVAL_TAU = 0.005

print(f"Loading and evaluating checkpoint:")
print(f"  Directory: {EVAL_CHECKPOINT_DIR}")
print(f"  Episode: {EVAL_EPISODE_NUM}")
print(f"  Grid Size: {EVAL_GRID_SIZE}")
print(f"  Eval Episodes: {EVAL_N_EVAL_EPISODES}")

# new env for inference
eval_env_for_init = BatchedSculpt3DEnvTF(grid_size=EVAL_GRID_SIZE, max_steps=EVAL_MAX_STEPS, n_envs=1)

loaded_agent = FeudalAgentTF(
    grid_shape=eval_env_for_init.grid_obs_shape,
    coord_shape=eval_env_for_init.coord_obs_shape,
    primitive_action_dim=6,
    manager_goal_k=EVAL_MANAGER_GOAL_K,
    subgoal_horizon=EVAL_SUBGOAL_HORIZON,
    intrinsic_reward_beta=EVAL_INTRINSIC_REWARD_BETA,
    manager_lr=EVAL_MANAGER_LR,
    worker_lr=EVAL_WORKER_LR,
    gamma=EVAL_GAMMA,
    tau=EVAL_TAU
    # buffer stuff is not needed for evaluation
)

manager_load_path = os.path.join(EVAL_CHECKPOINT_DIR, f"manager_ep{EVAL_EPISODE_NUM}_g{EVAL_GRID_SIZE}.weights.h5")
worker_load_path = os.path.join(EVAL_CHECKPOINT_DIR, f"worker_ep{EVAL_EPISODE_NUM}_g{EVAL_GRID_SIZE}.weights.h5")

try:
    loaded_agent.manager_model.load_weights(manager_load_path)
    loaded_agent.worker_model.load_weights(worker_load_path)
    print(f"Successfully loaded weights from:")
    print(f"  {manager_load_path}")
    print(f"  {worker_load_path}")

    # run eval
    evaluation_results = evaluate_agent_performance(
        agent=loaded_agent,
        grid_size=EVAL_GRID_SIZE,
        max_steps=EVAL_MAX_STEPS,
        num_eval_episodes=EVAL_N_EVAL_EPISODES,
        render=True, # set to True to render a final state
        render_env_index=0 # render final state
    )

except tf.errors.NotFoundError as e:
    print(f"\nError loading weights: {e}")
    print(f"Check that the directory '{EVAL_CHECKPOINT_DIR}' and episode number '{EVAL_EPISODE_NUM}' are correct.")
except Exception as e:
    print(f"\nAn error occurred during loading or evaluation: {e}")
    import traceback; traceback.print_exc()


      EVALUATING SAVED FEUDAL AGENT CHECKPOINT
Loading and evaluating checkpoint:
  Directory: checkpoints_feudal
  Episode: 10
  Grid Size: 8
  Eval Episodes: 50
Feudal Agent initialized:
  Subgoal Horizon (M): 10
  Intrinsic Reward Beta: 0.1
  Manager Goal K: 1 (Discrete goals: 27)
TensorBoard log directory: runs/feudal_dqn_20250430_145327
Successfully loaded weights from:
  checkpoints_feudal/manager_ep10_g8.weights.h5
  checkpoints_feudal/worker_ep10_g8.weights.h5

--- Running Evaluation (50 episodes) ---
  Initial number of carvable voxels: 376

--- Evaluation Results ---
  Avg Reward : -1190.54 (+/- 1125.39)
  Avg Length : 450.0
  Avg Removed: 17.1 / 376 (4.54% +/- 5.01%)
  Avg Incorrect: 0.0
  Evaluation Duration: 4.15s

--- Rendering final state for Eval Env Index: 0 ---
Saved evaluation render to renders_eval_feudal/eval_render_env_0_final.png
